# 15 — Scala for Spark

This is the bridge. Everything in notebooks 01 through 14 was the foundation; this notebook shows where each piece lands once you open a Spark codebase. The reason Spark is a *Scala* library — not just a JVM library — is that its API surface leans hard on language features you've now seen: case classes for schemas, pattern matching for row shapes, traits and generics for the type-safe Dataset API, implicits (now `given` and `using`) for everything from encoders to syntactic extensions, and lambdas for the transformations themselves.

You will not become a Spark expert in one notebook. The companion `apache-spark` track covers RDDs, DataFrames, execution model, joins, tuning, structured streaming, and Delta in depth. The goal here is narrower: when you open a Spark file and see `Dataset[Row]`, `import spark.implicits._`, `groupByKey`, or `udf((s: String) => ...)`, every line should look like *Scala you already know* — not like a separate dialect.

By the end you should be able to read idiomatic Scala-Spark code without surprise, and recognise which Spark constructs map back to which Scala features from earlier in this track.

## The three core types

Spark exposes three abstractions for distributed data. All three are ordinary Scala types — you reason about them with the intuitions you've already built.

- `RDD[A]` — a distributed, immutable collection of `A`. The lowest-level API. Closest in feel to `List[A]`, except partitioned across machines.
- `DataFrame` — a distributed table with named, typed columns. Internally `Dataset[Row]`. The most common type in modern code.
- `Dataset[A]` — a distributed collection of `A` where `A` is typically a case class. Combines the columnar engine of `DataFrame` with the type-safety of `RDD[A]`. The Scala-first API.

`DataFrame` is `Dataset[Row]` — it's a type alias. Read that twice; it explains why a lot of Spark documentation moves between the two names without warning.

In [ ]:
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset, Row}

val spark: SparkSession =
  SparkSession.builder.appName("notebook15").master("local[*]").getOrCreate()

// DataFrame is literally a type alias for Dataset[Row]
val df: DataFrame = spark.read.json("data.json")
val same: Dataset[Row] = df

## Case classes are schemas

A case class is the natural way to give a Spark row a name and a type. Spark inspects the case class at compile time, derives an `Encoder`, and uses it to do the binary serialisation that lets the columnar engine run at speed. The case-class fields become column names; the field types become column types.

Everything you learned about case classes in notebook 07 — copy, equality by value, pattern matching, destructuring — still applies. The case class isn't a separate Spark concept; it's the same Scala concept, with one extra superpower (Spark can derive a schema from it).

In [ ]:
case class Order(id: Long, userId: Long, amount: BigDecimal, currency: String)

import spark.implicits.*    // brings encoders for case classes into scope

val orders: Dataset[Order] =
  spark.read.parquet("orders").as[Order]

// Every transformation is type-checked end-to-end.
val highValue: Dataset[Order] = orders.filter(_.amount > BigDecimal(100))
val ids:       Dataset[Long]  = orders.map(_.id)

The `import spark.implicits.*` line is what notebook 11 called a `using` import — it brings into scope the encoders the compiler needs to type-check `as[Order]`, `map(_.id)`, and the `$"col"` column syntax further down. Without it, the compiler can't prove the type can be serialised, and you get a noisy error message. The fix is always: import the implicits.

## Pattern matching on rows

When you're working with untyped `DataFrame`s, each record is a `Row` — a sequence of values keyed by column name or position. Extracting fields by hand is verbose. Pattern matching, with the same `Row(...)` extractor shape you saw in notebook 08, makes it short.

In [ ]:
import org.apache.spark.sql.Row

val events: Dataset[Row] = spark.read.parquet("events")

val described: Dataset[String] = events.map {
  case Row(id: Long, "click", url: String) => s"click $id -> $url"
  case Row(id: Long, "view", _)            => s"view $id"
  case Row(id: Long, kind: String, _)      => s"other($kind) $id"
}

The match is the same `match` you used on `Option` and `Either` — Spark just provides the `Row` extractor. The `String` encoder needed for the resulting `Dataset[String]` is in scope thanks to the implicits import.

If you find yourself doing this a lot, that's the signal to move from `Dataset[Row]` to `Dataset[A]` for a case class `A`. Then field access is `.userId` instead of `case Row(id: Long, _, _)`, and the compiler tells you when the schema drifts.

## Familiar combinators, a different runtime

Most of the `Dataset` API will look familiar from notebooks 04 and 05 on collections.

- `map`, `flatMap`, `filter`, `groupBy`, `reduce`, `aggregate` — same names, same intent.
- The difference is *where* they run. A `List.map` runs on this thread, now. A `Dataset.map` runs on whatever executor holds each partition, eventually, when an action like `count`, `collect`, or `write` forces it.
- The shapes you write are nearly identical; the laziness is much deeper. A `Dataset` doesn't materialise until you ask for a result.

In [ ]:
// Looks like List ops; runs on the cluster.
val big:        Dataset[Order] = orders.filter(_.amount > BigDecimal(100))
val byCurrency: DataFrame      = orders.groupBy("currency").count()

// Nothing has been computed yet — these are descriptions.
big.write.parquet("big")          // now Spark runs the plan
byCurrency.show()                  // now Spark runs the plan

## `Option` meets nullability

Spark columns are nullable by default. Inside a `Dataset[A]`, the natural way to model "this field might be null" is `Option[T]` on the case class. The encoder handles the translation: `None` becomes a SQL `NULL`, `Some(v)` becomes `v`.

In [ ]:
case class User(id: Long, name: String, email: Option[String])

val users: Dataset[User] = spark.read.parquet("users").as[User]

// Handle absence the way you've handled it everywhere else.
val withEmail: Dataset[User]    = users.filter(_.email.isDefined)
val domains:   Dataset[String]  =
  users.flatMap(_.email.map(_.split("@").last))

Reaching for `row.getAs[String]("email")` on a raw `Row` is the brittle alternative. It returns `null` for missing fields, which propagates straight into a `NullPointerException` downstream — the exact failure mode `Option` was designed to remove. Stay in `Dataset[A]` whenever you can.

## `Either` for per-row error handling

A common need is *per-record validation* where some rows pass and some fail, and you want both — the clean stream to write downstream and the failed stream to send to a dead-letter location. The shape from notebook 09 maps directly: a pure Scala function returns `Either[Bad, Clean]`, and two `flatMap` passes split the source into typed good and bad outputs.

In [ ]:
import scala.util.Try

case class Raw(id: Long, amount: String)
case class Clean(id: Long, amount: BigDecimal)
case class Bad(id: Long, reason: String)

def parse(r: Raw): Either[Bad, Clean] =
  Try(BigDecimal(r.amount)).toEither
    .left.map(e => Bad(r.id, e.getMessage))
    .map(amt => Clean(r.id, amt))

val raw: Dataset[Raw] = spark.read.parquet("raw").as[Raw]

// Two passes; each one produces a typed Dataset of the appropriate case class.
val good: Dataset[Clean] = raw.flatMap(r => parse(r).toOption)
val bad:  Dataset[Bad]   = raw.flatMap(r => parse(r).left.toOption)

Two passes over `raw`, two outputs, one source. The `Either` is the contract between the validation step and the split — exactly the role it played in notebook 09, scaled out across partitions. If re-parsing twice would be expensive in your case, `raw.cache()` before the two `flatMap`s is the usual fix.

## Implicits and Spark, then and now

The reason you see `import spark.implicits.*` at the top of nearly every Scala-Spark file is that the Dataset API depends on type-class-style instances for two things: `Encoder[A]` (how to serialise `A`) and a handful of column-syntax helpers (the `$"col"` notation, for instance). In Scala 2 vocabulary these were `implicit val`s; in the Scala 3 vocabulary you've been using they are `given`s with `using` imports.

The Spark codebase itself is mid-migration in many places — you'll see both `implicit` and `given` in real code. The semantics are the same; the rules from notebook 11 apply unchanged.

## UDFs — the optimizer's blind spot

When the built-in column operations (`when`, `concat`, `regexp_extract`, the SQL functions) don't express what you need, you reach for a *user-defined function*. UDFs let you embed arbitrary Scala into the query, but the Catalyst optimizer cannot see inside them. To Spark, a UDF is an opaque box.

That has two consequences worth knowing before you write one. First, the optimizer cannot push filters or projections through a UDF, so a column you only needed to feed the UDF gets read in full. Second, the per-row Scala-to-internal-format conversion adds overhead the typed `Dataset[A]` API does not pay.

Prefer the typed `Dataset.map` for one-off logic, and built-in SQL functions whenever they exist. Reach for a UDF only when neither covers the case.

In [ ]:
import org.apache.spark.sql.functions.{udf, col}

// UDF — opaque, slower, no pushdown.
val upper = udf((s: String) => s.toUpperCase)
val df2   = users.toDF.withColumn("name_upper", upper(col("name")))

// Typed Dataset map — visible to the compiler, much faster for case-class shaped data.
val users2: Dataset[User] = users.map(u => u.copy(name = u.name.toUpperCase))

## Laziness, at the Spark scale

You met laziness in two earlier places. `lazy val` defers a single value's computation until the first read. Collection `view`s defer a chain of transformations until you force a result. `Dataset` is the third place — and the most important. Every `map`, `filter`, or `select` you write is *recorded* into a logical plan; nothing runs until an *action* like `count`, `collect`, `show`, or `write` triggers it.

That recording is what makes whole-pipeline optimisation possible. Catalyst sees the entire chain, rewrites it (predicate pushdown, projection pruning, join reordering), and then generates code. Calling actions in the middle of a pipeline — for debug `count`s especially — forces re-execution from the top and is a common cause of slow jobs.

## Futures and Spark

The Spark driver coordinates work, but each *action* on a `Dataset` blocks the calling thread until the job completes. You can wrap actions in `Future` to run multiple Spark queries concurrently against the same `SparkSession` — useful for unrelated reads or writes that share cluster capacity.

The pattern is unsurprising given notebook 13:

In [ ]:
import scala.concurrent.{Future, Await}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration.*

val a = Future { spark.read.parquet("orders").count() }
val b = Future { spark.read.parquet("users").count()  }

val (nOrders, nUsers) = Await.result(a.zip(b), 1.minute)

Both jobs are submitted to the same Spark scheduler and run in parallel against the cluster's resources. Don't use `Future` to parallelise transformations *within* a single `Dataset` — Spark already parallelises those across executors. Use it only when you want to run *multiple independent Spark jobs* at once.

## `SparkSession` as a resource

A `SparkSession` is a heavyweight resource — it holds connections to the cluster, caches metadata, and consumes driver memory. In tests, scripts, or short-lived jobs, give it a single owner and a clear shutdown. The patterns from notebook 14 apply directly.

In [ ]:
import scala.util.Using

given Using.Releasable[SparkSession] with
  def release(s: SparkSession): Unit = s.stop()

Using(SparkSession.builder.appName("ad-hoc").getOrCreate()) { spark =>
  spark.read.parquet("data").count()
}
// session is stopped on exit, even on failure

In a long-running service the lifetime is different — a single session for the life of the process, shut down on graceful termination. But in any one-shot context, the `Using` pattern keeps you from leaking the driver process. This is the same reasoning that made `Using` worth learning in notebook 14, applied at a different scale.

## Common Scala-meets-Spark pitfalls

The mistakes specific to the boundary, on top of the ones in notebook 14.

- **Forgetting `import spark.implicits.*`.** The error is usually "no encoder for X" or "value `$` is not a member of StringContext". The fix is always the import.
- **Capturing the driver in a closure.** A `Dataset.map` lambda is serialised and shipped to executors. If it captures a non-serialisable value from the surrounding scope — a `SparkSession`, a database connection, a logger — the task fails at runtime with a `TaskNotSerializable` error. Refactor to construct the value inside the lambda, or use `mapPartitions` so the construction cost is paid once per partition.
- **Calling actions in a loop.** Each action triggers a full re-execution of the plan. `dataset.cache()` is the lever when you really do need the same result twice.
- **Using UDFs when built-ins or typed `map` would do.** Covered above; worth repeating because it's the single most common performance smell in Scala-Spark code.
- **Confusing `Dataset[A]` with `RDD[A]`.** They have similar-looking APIs, but Dataset goes through Catalyst and RDD does not. New code should be Dataset (or DataFrame) unless you have a specific reason.
- **Mutable state in transformations.** A `var` inside a `map` lambda is meaningless across partitions — each executor sees its own copy. Use `aggregate`, `groupByKey`, or accumulators instead.

## Putting it together

A small but realistic pipeline. Read a JSON file of orders, parse them into a case class, route invalid rows to a side output, aggregate the rest by user, and write both results. Every construct here came from an earlier notebook.

In [ ]:
import org.apache.spark.sql.{SparkSession, Dataset}
import scala.util.{Try, Using}

case class RawOrder(id: Long, userId: Long, amount: String, currency: String)
case class Order(id: Long, userId: Long, amount: BigDecimal, currency: String)
case class BadOrder(id: Long, reason: String)
case class UserTotal(userId: Long, total: BigDecimal)

def parse(r: RawOrder): Either[BadOrder, Order] =
  Try(BigDecimal(r.amount)).toEither
    .left.map(e => BadOrder(r.id, e.getMessage))
    .map(amt => Order(r.id, r.userId, amt, r.currency))

given Using.Releasable[SparkSession] with
  def release(s: SparkSession): Unit = s.stop()

Using(SparkSession.builder.appName("orders").getOrCreate()) { spark =>
  import spark.implicits.*

  val raw = spark.read.json("orders.json").as[RawOrder].cache()

  val good: Dataset[Order]    = raw.flatMap(r => parse(r).toOption)
  val bad:  Dataset[BadOrder] = raw.flatMap(r => parse(r).left.toOption)

  val totals: Dataset[UserTotal] =
    good.groupByKey(_.userId)
        .reduceGroups((a, b) => a.copy(amount = a.amount + b.amount))
        .map { case (uid, o) => UserTotal(uid, o.amount) }

  totals.write.mode("overwrite").parquet("totals")
  bad.write.mode("overwrite").parquet("bad-orders")
}

Read that top to bottom. Case classes from notebook 07 model the schemas. `Try.toEither` from notebooks 09 and 14 captures parse failures. `Either` routes good and bad to separate outputs. `groupByKey` and `reduceGroups` from notebook 05's collection idioms scale to billions of records without changing shape. The `Using` block from notebook 14 owns the `SparkSession` lifetime.

There is no separate "Spark language" you needed to learn. There's Scala, and there are Spark types that participate in Scala's standard combinators. That's the whole point of the track — every feature notebooks 01 through 14 covered has a payoff in this last picture.

## What's next

This is the end of the Scala foundation. From here, the work splits in two directions, and you can take either or both.

- **Depth in Spark.** The companion `apache-spark` track picks up exactly where this notebook ended — DataFrames in detail, the execution model, joins and partitioning, Spark SQL and UDFs, performance tuning, structured streaming, Delta Lake. Open `apache-spark/01-spark-foundations.ipynb` to start.
- **Depth in Scala.** Cats, cats-effect, ZIO, fs2 — the ecosystem around pure functional programming and effect systems. None of it is required for productive Spark work, but learning at least one effect system is the single largest jump in *general* Scala fluency available after this track.

You have the whole language now. Go build something with it.